## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

## Reading in Data

In [3]:
data = pd.read_csv('../Data/exon_ranges_summary.csv')

## Simulation function

Defaults creating a CSV to False. (so don't get a ton of .csv's)

work in progress! 

In [16]:
def simulation(data, to_csv=False):
    mean_length = 5000 # avg. TE length
    std_dev_length = 2000 # std. dev. of TE length

    # num_TES = 100
    active_tes = random.randint(100, 10000) # letting randomness decide the constant* 

    te_lengths = np.random.normal(loc=mean_length, scale=std_dev_length, size=active_tes)

    te_lengths = np.clip(te_lengths, 100, 10000).astype(int)
    print(te_lengths)

    genome_size = data['Genome_size']
    average_range = data['Average Range']

    simulation_rounds = 1000 # but will allow more later
    te_move_threshold = 0.5 # probability threshold for TE mobilization

    range_1_start, range_1_end = 0, average_range
    range_2_start, range_2_end = average_range + 1, genome_size

    # Dictionary to store the results
    results = {
        'Species': data['Species'],
        'Beginning_GENOME_SIZE': genome_size,
        'Active_tes': active_tes,
        'range_1_start': range_1_start,
        'range_1_end': range_1_end,
        'range_2_start': range_2_start,
        'range_2_end': range_2_end,
        'TE_mobilized': 0,
        'TE_static': 0,
        'TE_in_exons': 0,
        'TE_in_non_coding': 0,
        'Exon_new_size': range_1_end - range_1_start + 1,
        'Non_coding_new_size': range_2_end - range_2_start + 1,
        'Total_Genome_growth': 0
    }

    for _ in range(simulation_rounds): # or active Te's ? 
        te_length = random.choice(te_lengths) # randomly select a TE length from normal distribution
        if random.random() < te_move_threshold:
            prob_exon = range_1_end / genome_size # probability of TE landing in exon
            # te_position = random.randint(0, genome_size)
            # print(f'TE position: {te_position}')

            if random.random() < prob_exon: # lands in exon (range 1)
                range_1_end += te_length # expand range 1
                range_2_start += range_1_end + 1 # adjust range 2 start
                range_2_end += te_length # expand range 2 
                results['TE_in_exons'] += 1
            else:
                range_2_end += te_length # expand range 2
                results['TE_in_non_coding'] += 1 # increment count for non-coding TEs
        
            results['TE_mobilized'] += 1
        else:
            results['TE_static'] += 1

    results['Exon_new_size'] = range_1_end - range_1_start + 1
    results['Non_coding_new_size'] = range_2_end - range_2_start + 1
    results['Total_Genome_growth'] = results['Exon_new_size'] + results['Non_coding_new_size'] - genome_size
    
        

    
    results_df = pd.DataFrame(results, index=[0])
    if to_csv:
        results_df.to_csv('CSV/simulation_results.csv', index=False)
    return results_df, te_lengths

## Run simulation for Sparrow Hawk. 

can do other species or all species, just change what is being passed to function. Change csv to true below if you would like to see the .csv

In [17]:
sparrow_hawk_df, sparrow_te_lengths = simulation(data.iloc[0], to_csv=False)
sparrow_hawk_df

[3708 4962 6869 ... 2460 6398 6533]


,Species,Beginning_GENOME_SIZE,Active_tes,range_1_start,range_1_end,range_2_start,range_2_end,TE_mobilized,TE_static,TE_in_exons,TE_in_non_coding,Exon_new_size,Non_coding_new_size,Total_Genome_growth
0,Accipiter_nisus.Accipiter_nisus_ver1.0.112,1190649881,1250,0,533365,533366,1190649881,488,512,0,488,533366,1192586029,2469514


## View TE lengths

In [24]:
np.set_printoptions(threshold=np.inf)
print(sparrow_te_lengths)


[ 3708  4962  6869  7200  2180  6570  5502  5473  7535  6015  5363  8229
  8342  2774  6942  5559  7329  3051  2437  6130  3344  5278  6688  6613
  4445  4807   100  8413  5022  3833  3450  7375  4868  8018  5121  2584
  4323  6466  5253  7545  4904  2741  5251  4088  7970  4400  6670  7446
  7999  4162  6098  6057  6052  6240  5783  5539  4933  3696  4732  6488
  4698  4007  6780  6434  5409  3565  4079  4319  3791  4228  5675  5568
   100  3269  6671  6161  5858  6379  4629  4182  6137  8171  1386  1612
  3294  5373  5761  8766  4474  6956  6184  7007  4037  5391  4482  6680
  4266  5330   827  5845  6292  5748  3987  4622  2352  6862  4469  1121
   962  1878  5510  3919  4342  7271  7743  7125  5798  7320  5774  1550
  8105  5947  6808  4040  7868  3893  7521  3958  7187  4169  4635  1168
  6267  9403  6385  3448  3985  4322  4033  7492  7011  3587  8409  4142
  4449  3995  6321  3335  4256  4417  5599  6981  8316  4973  5024  4050
  6252  5757  4389  3214  3287  2977  4098  8230  4